# <font color="steelblue">11. Valores atípicos, puntuaciones z y forma de la distribución</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 11 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`verificaciones.csv`, `salarios_redaccion.csv`, `audiencias_tv.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["verificaciones.csv", "salarios_redaccion.csv", "audiencias_tv.csv"]
PREGUNTAS = ["q_P11_1.json", "q_P11_2.json", "q_P11_3.json", "q_P11_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Un bulo compartido 160.000 veces cuando la mitad de los contenidos no pasa de 1.200. Un sueldo cinco veces mayor que el resto. Los **valores atípicos** pueden ser un error de los datos o **la noticia**. Antes de borrarlos o de destacarlos hay que detectarlos con criterio.

**Qué necesitas saber antes**: los cuadernos 9 y 10.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Detectar valores atípicos con la regla del 1,5 × RIC.
2. Calcular puntuaciones z para comparar valores de distribuciones distintas.
3. Medir la asimetría de una distribución.
4. Decidir qué hacer con un valor atípico.

## <font color='steelblue'>Contenidos</font>

1. La regla del 1,5 × RIC
2. Puntuaciones z
3. Asimetría
4. ¿Qué hago con un atípico?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. La regla del 1,5 × RIC</font>

In [ ]:
verif = pd.read_csv("verificaciones.csv")
salarios = pd.read_csv("salarios_redaccion.csv")

Un valor se considera **atípico** si queda por debajo de $Q_1 - 1{,}5 \cdot RIC$ o por encima de $Q_3 + 1{,}5 \cdot RIC$. Es la misma regla que usan los bigotes del diagrama de caja.

In [ ]:
def limites_atipicos(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    ric = q3 - q1
    return q1 - 1.5 * ric, q3 + 1.5 * ric

inf, sup = limites_atipicos(salarios["salario_bruto_anual"])
atipicos = salarios[(salarios["salario_bruto_anual"] < inf) | (salarios["salario_bruto_anual"] > sup)]
print(f"Límites: {inf:,.0f} € y {sup:,.0f} €")
print("Número de atípicos:", len(atipicos))
atipicos["puesto"].value_counts()

**Qué ha pasado**: los atípicos son sueldos de dirección, jefaturas de redacción y algún editor. No son errores: son sueldos (simulados) de otro tipo de puesto.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P11_1.json")

## <font color='steelblue'>2. Puntuaciones z</font>

La **puntuación z** dice a cuántas desviaciones típicas está un valor de la media:

$$z = \frac{x - \bar{x}}{s}$$

Sirve para comparar valores de distribuciones distintas: ¿es más excepcional un programa con 20 puntos de share o uno con 3.000.000 de espectadores? Valores con $|z| > 3$ suelen considerarse muy raros (en distribuciones aproximadamente simétricas).

In [ ]:
s = salarios["salario_bruto_anual"]
salarios["z"] = (s - s.mean()) / s.std()
salarios.sort_values("z", ascending=False)[["puesto", "salario_bruto_anual", "z"]].head().round(2)

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P11_2.json")

## <font color='steelblue'>3. Asimetría</font>

El **coeficiente de asimetría** resume la forma: vale 0 si es simétrica, es positivo si la cola larga va a la derecha y negativo si va a la izquierda. Las veces que se comparte un contenido en redes son un ejemplo extremo de asimetría positiva.

In [ ]:
print("Asimetría del salario:", round(salarios["salario_bruto_anual"].skew(), 2))
print("Asimetría de veces compartido:", round(verif["veces_compartido"].skew(), 2))
print(verif["veces_compartido"].describe().round(0))

Con una asimetría tan fuerte, el histograma normal no deja ver nada. Un truco habitual es usar una **escala logarítmica**, en la que cada marca del eje multiplica por 10:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(verif["veces_compartido"], bins=40, color="grey")
axes[0].set_title("Escala normal")
axes[1].hist(np.log10(verif["veces_compartido"]), bins=40, color="steelblue")
axes[1].set_title("Escala logarítmica (10³ = mil, 10⁴ = diez mil...)")
axes[1].set_xlabel("log10(veces compartido)")
plt.tight_layout()
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P11_3.json")

## <font color='steelblue'>4. ¿Qué hago con un atípico?</font>

Antes de tocar un valor atípico pregúntate **por qué está ahí**:

| Situación | Qué hacer |
| :-- | :-- |
| Error de tecleo o de unidades (un sueldo de 2.500.000 €) | Comprobar con la fuente y corregir |
| Caso de otro tipo (la dirección en una tabla de redactores) | Analizarlo aparte |
| Caso real y extremo (un bulo viral) | **Es la noticia**: no lo borres; usa mediana y percentiles para el resto |

Nunca elimines datos solo porque «estropean» el resultado, y si los excluyes, dilo en la nota metodológica.

In [ ]:
print("Bulo más compartido:")
verif.sort_values("veces_compartido", ascending=False).head(3)

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P11_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Aplica la regla del 1,5 × RIC a `veces_compartido`. ¿Qué porcentaje de contenidos son atípicos?

<details><summary>Ver solución</summary>

```python
inf, sup = limites_atipicos(verif["veces_compartido"])
round(((verif["veces_compartido"] > sup) | (verif["veces_compartido"] < inf)).mean() * 100, 1)
```
</details>

### **Ejercicio 2** *(sección 1)*

¿Qué veredicto (Falso, Engañoso, Verdadero) tienen los contenidos atípicamente compartidos?

<details><summary>Ver solución</summary>

```python
verif[verif["veces_compartido"] > sup]["veredicto"].value_counts(normalize=True)
```
</details>

### **Ejercicio 3** *(sección 2)*

Calcula la puntuación z de un programa con 20 puntos de share y de otro con 3.000 (miles) de espectadores, usando `audiencias_tv.csv`. ¿Cuál es más excepcional?

<details><summary>Ver solución</summary>

```python
aud = pd.read_csv("audiencias_tv.csv")
z1 = (20 - aud["share"].mean()) / aud["share"].std()
z2 = (3000 - aud["espectadores_miles"].mean()) / aud["espectadores_miles"].std()
print(z1, z2)
```
</details>

### **Ejercicio 4** *(sección 3)*

Compara la asimetría de las veces compartido según el veredicto.

<details><summary>Ver solución</summary>

```python
verif.groupby("veredicto")["veces_compartido"].skew()
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Un valor es **atípico** si queda fuera de $[Q_1 - 1{,}5\,RIC,\; Q_3 + 1{,}5\,RIC]$.
- La **puntuación z** indica a cuántas desviaciones típicas de la media está un valor y permite comparar escalas distintas.
- El **coeficiente de asimetría** es 0 si la distribución es simétrica y positivo si la cola va a la derecha; con asimetrías fuertes ayuda la **escala logarítmica**.
- Un atípico puede ser un **error**, un **caso de otro grupo** o **la noticia**: compruébalo antes de decidir.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Valor atípico** | Dato muy alejado del resto. |
| **Puntuación z** | Distancia a la media medida en desviaciones típicas. |
| **Coeficiente de asimetría** | Medida numérica de la falta de simetría. |
| **Escala logarítmica** | Escala en la que cada marca multiplica por 10. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `np.log10` da `-inf` | Hay valores 0 | Usa `np.log10(x + 1)` o filtra los ceros |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 12 empezarás a estudiar relaciones entre variables: tablas de contingencia.

**Siguiente cuaderno**: [12. Relación entre variables cualitativas: tablas de contingencia](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P12_tablas_de_contingencia.ipynb)